In [14]:
import cv2
import pandas as pd
import numpy as np

image_path = "images/"

regions = []

for j in range(0, 3):
    for i in range(0, 3):
        regions.append([i * 64, (i + 1) * 64 + 64, j * 64, (j + 1) * 64 + 64])

hog = cv2.HOGDescriptor(
    _winSize=(64, 128),
    _blockSize=(16, 16),
    _blockStride=(8, 8),
    _cellSize=(8, 8),
    _nbins=9
)

def extract_hog_features(image):
    features = []
    for region in regions:
        x1, x2, y1, y2 = region
        mid = x1 + 63
        im_win1 = image[y1:y2, x1:mid]
        im_win2 = image[y1:y2, mid+1:x2]
        win_features1 = hog.compute(im_win1).flatten()
        win_features2 = hog.compute(im_win2).flatten()
        features.append(np.concatenate([win_features1, win_features2]))
    return features

# train için
train_frames = pd.read_csv('frames_train.csv')
train_features = []
for i in range(len(train_frames)):
    image = cv2.imread(image_path + train_frames.loc[i]['frame'])
    feats = extract_hog_features(image)
    train_features.append(feats)

train_frames['features'] = train_features
train_frames.to_csv('frames_train_with_features.csv', index=False)


# validation için
val_frames = pd.read_csv('frames_val.csv')
val_features = []
for i in range(len(val_frames)):
    image = cv2.imread(image_path + val_frames.loc[i]['frame'])
    feats = extract_hog_features(image)
    val_features.append(feats)

val_frames['features'] = val_features
val_frames.to_csv('frames_val_with_features.csv', index=False)

# test için
test_frames = pd.read_csv('frames_test.csv')
test_features = []
for i in range(len(test_frames)):
    image = cv2.imread(image_path + test_frames.loc[i]['frame'])
    feats = extract_hog_features(image)
    test_features.append(feats)

test_frames['features'] = test_features
test_frames.to_csv('frames_test_with_features.csv', index=False)

